# 02 · Single Stock Valuation

Run the full three-model ensemble on one ticker and inspect every layer:

1. **DCF waterfall** — where enterprise value comes from
2. **FCF projection** — historical trend + 10-year model forecast
3. **Sensitivity heatmap** — how fair value moves with WACC and growth
4. **Three-model comparison** — DCF / Relative / ML side by side
5. **Verdict** — margin of safety and confidence interpretation
6. **Sentiment overlay** — NLP signals and how they shift the confidence score

In [1]:
import sys
import pathlib

sys.path.insert(0, str(pathlib.Path().resolve().parent))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from dotenv import load_dotenv
load_dotenv()

from fairprice.data import FinancialsClient, MarketClient, MacroClient
from fairprice.models import dcf
from fairprice.nlp import SentimentClient
import fairprice.valuation as valuation_engine

TICKER = "AAPL"  # change to any ticker

fin = FinancialsClient()
mkt = MarketClient()
mac = MacroClient()
nlp = SentimentClient()
print("Clients ready.")

Clients ready.


In [2]:
profile = fin.get_profile(TICKER)
stmts = fin.get_statements(TICKER)
market = mkt.get_market_data(TICKER)
macro = mac.get_macro_data()
macro.vix = market.vix  # use live VIX
peers = fin.get_peers(TICKER)

print(f"{profile.name}  |  {profile.sector}  |  {profile.currency}")
print(f"Market cap : ${market.market_cap/1e9:.0f}B")
print(f"Price      : ${market.current_price:.2f}")
print(f'Peers      : {peers}')

Apple Inc.  |  Technology  |  USD
Market cap : $4566B
Price      : $310.85
Peers      : [{'symbol': 'GOOGL', 'companyName': 'Alphabet Inc.', 'price': 388.83, 'mktCap': 4702848416027}, {'symbol': 'META', 'companyName': 'Meta Platforms, Inc.', 'price': 635.255, 'mktCap': 1612546176025}, {'symbol': 'MSFT', 'companyName': 'Microsoft Corporation', 'price': 412.67, 'mktCap': 3065490208100}, {'symbol': 'NVDA', 'companyName': 'NVIDIA Corporation', 'price': 212.6, 'mktCap': 5149384600000}, {'symbol': 'NXT', 'companyName': 'Nextpower Inc.', 'price': 135.78, 'mktCap': 20404267808}, {'symbol': 'RIME', 'companyName': 'Algorhythm Holdings, Inc.', 'price': 0.7312, 'mktCap': 1809864}, {'symbol': 'SONY', 'companyName': 'Sony Group Corporation', 'price': 21.86, 'mktCap': 129150222729}, {'symbol': 'TBCH', 'companyName': 'Turtle Beach Corporation', 'price': 12.7, 'mktCap': 252061980}, {'symbol': 'TSM', 'companyName': 'Taiwan Semiconductor Manufacturing Company Limited', 'price': 422.775, 'mktCap': 2192722

## 1 · DCF Waterfall

Enterprise value = PV(stage-1 FCFs) + PV(stage-2 FCFs) + PV(terminal value).  
Subtract net debt to reach equity value, then divide by shares outstanding.

In [3]:
comp = dcf.get_components(stmts, market, macro)

if comp is None:
    print("DCF skipped — no positive FCF found.")
else:
    # Waterfall data
    labels = [
        "PV Stage 1\n(Yrs 1–5)",
        "PV Stage 2\n(Yrs 6–10)",
        "PV Terminal",
        "Net Debt (−)",
        "Equity Value",
    ]
    values = [
        comp.pv_stage1 / 1e9,
        comp.pv_stage2 / 1e9,
        comp.pv_terminal / 1e9,
        -comp.net_debt / 1e9,
        0,
    ]
    measures = ["relative", "relative", "relative", "relative", "total"]

    tv_pct = comp.pv_terminal / comp.enterprise_value * 100

    fig = go.Figure(
        go.Waterfall(
            name="DCF",
            orientation="v",
            measure=measures,
            x=labels,
            y=values,
            text=[
                f"${v:+.0f}B" if m != "total" else f"${comp.equity_value/1e9:.0f}B"
                for v, m in zip(values, measures)
            ],
            textposition="outside",
            connector=dict(line=dict(color="#9E9E9E", width=1, dash="dot")),
            increasing=dict(marker=dict(color="#4CAF50")),
            decreasing=dict(marker=dict(color="#F44336")),
            totals=dict(marker=dict(color="#2196F3")),
        )
    )

    fig.update_layout(
        title=f"{profile.name} DCF Decomposition  "
        f"(Terminal value = {tv_pct:.0f}% of EV)",
        yaxis_title="$B",
        height=440,
    )
    fig.show()

    print(f"Base FCF         : ${comp.base_fcf/1e9:.1f}B")
    print(f"Stage-1 growth   : {comp.growth_stage1*100:.1f}% p.a.")
    print(f"Terminal growth  : {comp.terminal_growth*100:.1f}%")
    print(f"WACC             : {comp.wacc*100:.2f}%")
    print(f"Enterprise value : ${comp.enterprise_value/1e9:.0f}B")
    print(f"Net debt         : ${comp.net_debt/1e9:.1f}B")
    print(f"Fair value/share : ${comp.per_share:.2f}")

Base FCF         : $129.2B
Stage-1 growth   : -3.9% p.a.
Terminal growth  : 4.0%
WACC             : 9.61%
Enterprise value : $1509B
Net debt         : $62.7B
Fair value/share : $98.47


## 2 · FCF Projection

Historical FCF alongside the base, bear, and bull projections used by the model.

In [4]:
from fairprice.models.dcf import _base_fcf, _growth_estimates, _terminal_growth

base_fcf_val = _base_fcf(stmts)
g_bear, g_base, g_bull = _growth_estimates(stmts)
g_term = _terminal_growth(macro)

# Historical FCF
cf = stmts.cash_flow
hist_fcf, hist_years = [], []
for col in ["Free Cash Flow", "Operating Cash Flow"]:
    if col in cf.columns:
        s = cf[col].dropna()
        hist_fcf = (s / 1e9).tolist()
        hist_years = [int(d.strftime("%Y")) for d in s.index]
        break

# Project 10 years
STAGE1, STAGE2 = 5, 5


def _project(base, growth):
    vals, fcf = [], base
    for yr in range(1, STAGE1 + 1):
        fcf *= 1 + growth
        vals.append(fcf / 1e9)
    for step in range(1, STAGE2 + 1):
        fade = growth + (g_term - growth) * step / STAGE2
        fcf *= 1 + fade
        vals.append(fcf / 1e9)
    return vals


last_hist_yr = int(hist_years[-1]) if hist_years else 2023
proj_years = [str(last_hist_yr + i) for i in range(1, 11)]

base_proj = _project(base_fcf_val, g_base)
bear_proj = _project(base_fcf_val, g_bear)
bull_proj = _project(base_fcf_val, g_bull)

fig = go.Figure()

# Historical
fig.add_trace(
    go.Scatter(
        x=hist_years,
        y=hist_fcf,
        name="Historical FCF",
        mode="lines+markers",
        line=dict(color="#2196F3", width=2.5),
        marker=dict(size=8),
    )
)

# Bull/bear shading
fig.add_trace(
    go.Scatter(
        x=proj_years + proj_years[::-1],
        y=bull_proj + bear_proj[::-1],
        fill="toself",
        fillcolor="rgba(76,175,80,0.12)",
        line=dict(color="rgba(0,0,0,0)"),
        name="Bear–Bull range",
        showlegend=True,
    )
)

# Projections
for name, proj, color, dash in [
    ("Bear", bear_proj, "#F44336", "dot"),
    ("Base", base_proj, "#4CAF50", "solid"),
    ("Bull", bull_proj, "#2196F3", "dash"),
]:
    fig.add_trace(
        go.Scatter(
            x=[hist_years[-1]] + proj_years if hist_years else proj_years,
            y=([hist_fcf[-1]] + proj if hist_fcf else proj),
            name=f'{name} ({(g_base if name=="Base" else g_bear if name=="Bear" else g_bull)*100:.0f}%)',
            mode="lines",
            line=dict(color=color, width=2, dash=dash),
        )
    )

fig.add_vline(
    x=hist_years[-1] if hist_years else proj_years[0],
    line_dash="dash",
    line_color="gray",
    annotation_text="Forecast →",
)
fig.update_layout(
    title="Free Cash Flow: Historical + 10-Year Projection ($B)",
    yaxis_title="$B",
    height=420,
)
fig.show()

## 3 · Sensitivity Heatmap

A DCF is only as good as its assumptions. This heatmap shows how the  
**base fair value per share** changes across a WACC × growth grid.  
The model's actual WACC and growth estimate are highlighted with a marker.

In [5]:
from fairprice.models.dcf import _scenario, _net_debt as _nd, _terminal_growth

if comp is None or base_fcf_val is None:
    print("DCF not available — skipping sensitivity.")
else:
    waccs = [0.07, 0.08, 0.09, 0.10, 0.11, 0.12, 0.13]
    growths = [0.04, 0.06, 0.08, 0.10, 0.12, 0.15, 0.18]
    g_term = _terminal_growth(macro)

    matrix = []
    for w in waccs:
        row = []
        for g in growths:
            val = _scenario(base_fcf_val, g, w, g_term, stmts, market)
            row.append(round(val, 2))
        matrix.append(row)

    wacc_labels = [f"{w*100:.0f}%" for w in waccs]
    growth_labels = [f"{g*100:.0f}%" for g in growths]

    # Annotate cells: green if > price, red if < price
    text_matrix = [[f"${v:.0f}" for v in row] for row in matrix]

    fig = go.Figure(
        go.Heatmap(
            z=matrix,
            x=growth_labels,
            y=wacc_labels,
            text=text_matrix,
            texttemplate="%{text}",
            textfont=dict(size=11),
            colorscale="RdYlGn",
            zmid=market.current_price,
            colorbar=dict(title="Fair Value $"),
        )
    )

    # Mark model's assumptions
    wacc_idx = min(range(len(waccs)), key=lambda i: abs(waccs[i] - comp.wacc))
    growth_idx = min(
        range(len(growths)), key=lambda i: abs(growths[i] - comp.growth_stage1)
    )

    fig.add_trace(
        go.Scatter(
            x=[growth_labels[growth_idx]],
            y=[wacc_labels[wacc_idx]],
            mode="markers",
            marker=dict(
                symbol="star",
                size=16,
                color="white",
                line=dict(color="black", width=1.5),
            ),
            name="Model estimate",
            showlegend=True,
        )
    )

    fig.update_layout(
        title=f"Fair Value Sensitivity  (current price = ${market.current_price:.0f}  |  "
        f"green cells = above market price)",
        xaxis_title="Stage-1 FCF Growth Rate",
        yaxis_title="WACC",
        height=440,
    )
    fig.show()

    above = sum(v > market.current_price for row in matrix for v in row)
    total = len(waccs) * len(growths)
    print(
        f"{above}/{total} scenarios ({above/total:.0%}) produce a fair value "
        f"above the current market price of ${market.current_price:.2f}"
    )

13/49 scenarios (27%) produce a fair value above the current market price of $310.85


## 4 · Three-Model Comparison

Each model attacks fair value differently.  
- **High agreement** among models → higher confidence in the estimate  
- **Large spread** → fundamental uncertainty; margin of safety should be wider

In [6]:
result = valuation_engine.estimate(stmts, market, macro, peers)

model_vals = {}
if result.dcf_value:
    model_vals["DCF"] = result.dcf_value
if result.relative_value:
    model_vals["Relative"] = result.relative_value
if result.ml_value:
    model_vals["ML Forecast"] = result.ml_value

names = list(model_vals.keys()) + ["Current Price", "Fair Value (base)"]
values = list(model_vals.values()) + [market.current_price, result.fair_value_base]
colors = ["#2196F3"] * len(model_vals) + [
    "#9E9E9E",
    "#FF9800" if result.margin_of_safety > 0 else "#F44336",
]

fig = go.Figure(
    go.Bar(
        x=names,
        y=values,
        marker_color=colors,
        text=[f"${v:.2f}" for v in values],
        textposition="outside",
    )
)

fig.add_hline(
    y=market.current_price,
    line_dash="dash",
    line_color="#9E9E9E",
    annotation_text=f"Market price ${market.current_price:.2f}",
)

if len(model_vals) > 1:
    vals = list(model_vals.values())
    fig.add_hrect(
        y0=min(vals),
        y1=max(vals),
        fillcolor="rgba(33,150,243,0.07)",
        line_width=0,
        annotation_text="Model range",
    )

fig.update_layout(
    title=f"{profile.name} — Valuation Model Comparison",
    yaxis_title="Per-Share Value ($)",
    height=420,
)
fig.show()

if len(model_vals) > 1:
    spread = (
        max(model_vals.values()) - min(model_vals.values())
    ) / result.fair_value_base
    print(
        f"Model spread: {spread:.0%} of base fair value "
        f'({"tight — good agreement" if spread < 0.20 else "wide — high model uncertainty"})'
    )

Model spread: 19% of base fair value (tight — good agreement)


## 5 · Verdict — Margin of Safety & Confidence

In [7]:
mos_pct = result.margin_of_safety * 100
conf = result.confidence_score

# Bull/bear/base range chart
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=[result.fair_value_low, result.fair_value_base, result.fair_value_high],
        y=["Fair Value", "Fair Value", "Fair Value"],
        mode="markers",
        name="Fair value range",
        marker=dict(
            size=[12, 20, 12],
            color=["#FF9800", "#4CAF50" if mos_pct > 0 else "#F44336", "#FF9800"],
            symbol=["triangle-left", "circle", "triangle-right"],
        ),
    )
)

fig.add_vline(
    x=market.current_price,
    line_dash="solid",
    line_color="#9E9E9E",
    line_width=2,
    annotation_text=f"Market ${market.current_price:.2f}",
    annotation_position="top",
)

fig.add_shape(
    type="rect",
    x0=result.fair_value_low,
    x1=result.fair_value_high,
    y0=-0.5,
    y1=0.5,
    fillcolor="rgba(76,175,80,0.15)",
    line=dict(color="#4CAF50", dash="dot"),
)

fig.update_layout(
    title=f"Fair Value Range — MoS = {mos_pct:+.1f}%  |  Confidence = {conf:.2f}",
    xaxis_title="Per-Share Value ($)",
    yaxis=dict(showticklabels=False),
    height=280,
)
fig.show()

# ── Confidence gauge ──────────────────────────────────────────────────────────
fig2 = go.Figure(
    go.Indicator(
        mode="gauge+number+delta",
        value=conf,
        title=dict(text="Confidence Score"),
        delta=dict(reference=0.5, increasing=dict(color="#4CAF50")),
        gauge=dict(
            axis=dict(range=[0, 1]),
            bar=dict(color="#2196F3"),
            steps=[
                dict(range=[0.0, 0.4], color="#FFCDD2"),
                dict(range=[0.4, 0.6], color="#FFF9C4"),
                dict(range=[0.6, 1.0], color="#C8E6C9"),
            ],
            threshold=dict(line=dict(color="red", width=3), value=0.4),
        ),
    )
)
fig2.update_layout(height=300)
fig2.show()

# ── Actionable summary ────────────────────────────────────────────────────────
print("=" * 56)
print(f"  {profile.name}  ({TICKER})")
print("=" * 56)
print(f"  Market price      : ${market.current_price:.2f}")
print(f"  Fair value (base) : ${result.fair_value_base:.2f}")
print(
    f"  Range             : ${result.fair_value_low:.2f} – ${result.fair_value_high:.2f}"
)
print(f"  Margin of safety  : {mos_pct:+.1f}%")
print(f"  Confidence        : {conf:.2f}")
print()

if conf < 0.35:
    print("  ⚠  LOW CONFIDENCE — do not act on this estimate alone.")
    print("     Check data completeness in notebook 01.")
elif mos_pct > 20:
    print("  ★  POTENTIALLY UNDERVALUED")
    print(f"     Stock trades {mos_pct:.0f}% below our base fair value.")
    print(f"     Bull scenario offers ${result.fair_value_high:.2f} upside.")
    print(
        f"     Bear scenario still implies ${result.fair_value_low:.2f} "
        f"({(result.fair_value_low/market.current_price - 1)*100:+.0f}% vs market)."
    )
elif mos_pct < -20:
    print("  ▼  POTENTIALLY OVERVALUED")
    print(f"     Stock trades {abs(mos_pct):.0f}% above our base fair value.")
    print(f"     Current valuation requires {g_bull*100:.0f}%+ FCF growth to justify.")
    print(f"     Check sensitivity heatmap for break-even WACC/growth combinations.")
else:
    print("  ~  FAIRLY VALUED  (within ±20% of base estimate)")
    print(f"     No strong directional signal. Watch for earnings revisions.")

if result.notes:
    print()
    print("  Model notes:")
    for note in result.notes:
        print(f"    • {note}")

  Apple Inc.  (AAPL)
  Market price      : $310.85
  Fair value (base) : $106.23
  Range             : $95.63 – $114.58
  Margin of safety  : -192.6%
  Confidence        : 0.61

  ▼  POTENTIALLY OVERVALUED
     Stock trades 193% above our base fair value.
     Current valuation requires -5%+ FCF growth to justify.
     Check sensitivity heatmap for break-even WACC/growth combinations.

  Model notes:
    • Relative: no peers or insufficient data — skipped


## 6 · Sentiment Overlay

The NLP layer aggregates news from **yfinance**, **FinViz**, and **Alpaca News**, then scores
each headline using the active backend (FinBERT → VADER → keyword lexicon — auto-selected at
import based on installed packages).

> Sentiment adjusts the **confidence score only** (±10 % maximum), never the fair value
> itself — short-term news is too noisy to move a fundamental estimate, but it can
> legitimately raise or lower our *conviction* in the model output.

| Signal | Condition | Confidence multiplier |
|--------|-----------|----------------------|
| Bearish + uncertain | score < −0.3 AND uncertainty > 0.5 | × 0.90 |
| Guidance lowered | management lowered guidance | × 0.95 |
| Bullish | score > +0.4 | × 1.05 |
| Guidance raised | management raised guidance | × 1.03 |

In [8]:
from fairprice.nlp.sentiment import active_backend

print(f"Fetching sentiment for {TICKER}  [backend: {active_backend()}]...")
sent = nlp.get_sentiment(TICKER)

# Re-run valuation with sentiment so we can compare confidence before/after
result_sent = valuation_engine.estimate(stmts, market, macro, peers, sentiment=sent)

print(f"\nSources        : {', '.join(sent.sources) or 'none'}")
print(f"Articles       : {sent.n_articles} (yfinance + FinViz)  |  Alpaca: {sent.n_alpaca_articles}")
print(f"\nSentiment score: {sent.sentiment_score:+.4f}  (−1 = very bearish … +1 = very bullish)")
print(f"Positive ratio : {sent.positive_ratio:.1%}")
print(f"Negative ratio : {sent.negative_ratio:.1%}")
print(f"Neutral  ratio : {sent.neutral_ratio:.1%}")
print(f"Uncertainty    : {sent.uncertainty_score:.4f}  (0 = clear … 1 = very uncertain)")
if sent.guidance_revision:
    print(f"Guidance       : {sent.guidance_revision.upper()}")
if sent.analyst_consensus:
    print(f"Analyst view   : {sent.analyst_consensus.upper()}")
conf_delta = result_sent.confidence_score - result.confidence_score
arrow = "▲" if conf_delta > 0.001 else "▼" if conf_delta < -0.001 else "─"
print(f"\nConfidence  before sentiment : {result.confidence_score:.4f}")
print(f"Confidence  after  sentiment : {result_sent.confidence_score:.4f}  {arrow} {conf_delta:+.4f}")

Fetching sentiment for AAPL  [backend: keyword]...

Sources        : finviz, alpaca
Articles       : 20 (yfinance + FinViz)  |  Alpaca: 30

Sentiment score: +0.1560  (−1 = very bearish … +1 = very bullish)
Positive ratio : 28.5%
Negative ratio : 12.9%
Neutral  ratio : 58.6%
Uncertainty    : 0.0449  (0 = clear … 1 = very uncertain)
Guidance       : REITERATED

Confidence  before sentiment : 0.6122
Confidence  after  sentiment : 0.6122  ─ +0.0000


In [9]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"{TICKER} — Positive / Neutral / Negative breakdown",
        "Confidence: before vs after sentiment",
    ],
    column_widths=[0.55, 0.45],
)

for label, val, color in [
    ("Positive", sent.positive_ratio, "#4CAF50"),
    ("Neutral",  sent.neutral_ratio,  "#9E9E9E"),
    ("Negative", sent.negative_ratio, "#F44336"),
]:
    fig.add_trace(
        go.Bar(
            name=label, x=[val], y=[TICKER], orientation="h",
            marker_color=color, text=[f"{val:.0%}"],
            textposition="inside", insidetextanchor="middle",
        ),
        row=1, col=1,
    )

fig.add_trace(
    go.Bar(
        x=["Before sentiment", "After sentiment"],
        y=[result.confidence_score, result_sent.confidence_score],
        marker_color=["#90CAF9", "#2196F3"],
        text=[f"{v:.3f}" for v in [result.confidence_score, result_sent.confidence_score]],
        textposition="outside", showlegend=False,
    ),
    row=1, col=2,
)

score_label = (
    "▼ BEARISH" if sent.sentiment_score < -0.3
    else "▲ BULLISH" if sent.sentiment_score > 0.3
    else "~ NEUTRAL"
)
fig.add_annotation(
    x=0.5, y=-0.42, xref="x domain", yref="y domain",
    text=f"Net score: {sent.sentiment_score:+.3f}  {score_label}  |  Uncertainty: {sent.uncertainty_score:.2f}",
    showarrow=False, font=dict(size=12), row=1, col=1,
)

fig.update_layout(
    barmode="stack", height=330,
    title=f"{profile.name} — Sentiment Analysis  [{sent.backend} backend]",
    legend=dict(orientation="h", y=1.12),
)
fig.update_xaxes(range=[0, 1], row=1, col=1)
fig.update_yaxes(range=[0, 1.15], row=1, col=2)
fig.show()

In [10]:
from fairprice.nlp.sentiment import score_texts, detect_uncertainty

# ── Per-headline scores ──────────────────────────────────────────────────────
if sent.top_headlines:
    texts  = sent.top_headlines
    scores = score_texts(texts)
    print(f"Recent headlines for {TICKER}  ({len(texts)} shown):")
    print(f"{'':3}  {'Pos':>5}  {'Neg':>5}  {'Unc':>5}  Headline")
    print("─" * 90)
    for h, s in zip(texts, scores):
        net = s["positive"] - s["negative"]
        unc = detect_uncertainty(h)
        sig = "📈" if net > 0.15 else "📉" if net < -0.15 else "➖"
        print(f"{sig}   {s['positive']:>4.0%}  {s['negative']:>4.0%}  {unc:>4.2f}  {h[:75]}")
else:
    print("No live headlines — configure API keys in .env to fetch live data.")

# ── Combined signal interpretation ──────────────────────────────────────────
print(f"\n{'═'*62}")
print(f"  Combined signal: {TICKER}")
print(f"{'═'*62}")
mos_pct = result_sent.margin_of_safety * 100
ss      = sent.sentiment_score

if mos_pct > 15 and ss > 0.1:
    verdict = "★  STRONG BUY    — undervalued + positive news flow"
elif mos_pct > 15 and ss < -0.1:
    verdict = "⚡  VALUE TRAP?   — cheap fundamentally, bearish news flow"
elif mos_pct < -15 and ss < -0.1:
    verdict = "▼  CLEAR AVOID   — overvalued + negative news flow"
elif mos_pct < -15 and ss > 0.1:
    verdict = "⚠  MOMENTUM PLAY? — expensive but positive near-term flow"
else:
    verdict = "~  MIXED SIGNAL   — no strong directional consensus"

print(f"  {verdict}")
print(f"\n  Margin of safety  : {mos_pct:+.1f}%")
print(f"  Sentiment score   : {ss:+.3f}")
print(f"  Confidence score  : {result_sent.confidence_score:.3f}")
if result_sent.notes:
    print("\n  Sentiment notes:")
    for n in result_sent.notes:
        if "sentiment" in n.lower() or "Sentiment" in n:
            print(f"    • {n}")

Recent headlines for AAPL  (5 shown):
       Pos    Neg    Unc  Headline
──────────────────────────────────────────────────────────────────────────────────────────
➖     0%    0%  0.00  Financial Times Reported Nvidia CEO Jensen Huang To Join Advisory Board At 
📉     0%   90%  0.00  Wall Street Rally Rests On 'Very Narrow Subset Of Stocks,' Analyst Warns— I
➖     0%    0%  0.00  Jensen Huang Joins Elite Tsinghua University Advisory Board That Includes T
📈    90%    0%  0.00  South Korea Is Crushing The Nasdaq 100 By Most Margin Since 2001. The EWY-t
📈    90%    0%  0.00  What's Going On With Broadcom Stock Wednesday?. AVGO preview: Broadcom sets

══════════════════════════════════════════════════════════════
  Combined signal: AAPL
══════════════════════════════════════════════════════════════
  ⚠  MOMENTUM PLAY? — expensive but positive near-term flow

  Margin of safety  : -192.6%
  Sentiment score   : +0.156
  Confidence score  : 0.612

  Sentiment notes:
